In [20]:
from pathlib import Path
import joblib, json
import numpy as np
import pandas as pd

PROJ = Path.cwd().parents[0]
MODELS = PROJ / "models"
DATA_RAW = PROJ / "data" / "raw"
REPORTS = PROJ / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

pre = joblib.load(MODELS / "preprocessor.joblib")
cands = sorted(MODELS.glob("*_tuned_calibrated.joblib"))
model_path = cands[0] if cands else sorted(MODELS.glob("*.joblib"))[0]
model = joblib.load(model_path)
print("Loaded preprocessor:", MODELS / "preprocessor.joblib")
print("Loaded model:", model_path.name)

Loaded preprocessor: d:\Git\DepressionLevel\DepressionModelProject\models\preprocessor.joblib
Loaded model: GradientBoosting_tuned_calibrated.joblib


In [21]:
num_cols = pre.transformers_[0][2]
cat_cols = pre.transformers_[1][2]
raw_input_columns = list(num_cols) + list(cat_cols)
print("Expected input columns (raw):", len(raw_input_columns))
raw_input_columns

Expected input columns (raw): 13


['Timestamp',
 'What is Your Age group?',
 'What is Your Gender',
 '1. Have You Been Feeling Sad Most of The Time During The Day, Especially Over The Last Two Weeks ?',
 '2. Do You Feel Unhappy Even When You Do Your Favorite Activities ?',
 '3. Do You Feel Tired All The Time , Do You Find It Difficulty to Complete Day Today Usual Activity ?',
 '4. Do you often catch yourself getting distracted while working on something important?',
 "5. Do you often feel that you're not as capable as your friends, even when you try your best?",
 "6. Do you often feel guilty about things that aren't your fault or feel worthless?",
 '7. Do you feel as if things won’t get better no matter what you do?',
 '8. Lately, have you felt so overwhelmed or hopeless that you thought about giving up on everything and even Life?',
 '9. Have you been having trouble falling asleep, waking early, or sleeping too much?',
 '10. Have you lost your appetite or found that you’re eating much less than usual?']

In [22]:
# Single-record prediction from a dict
row = {
   
    "Timestamp": "2025-10-09 10:00",
    "What is Your Age group?": "25-34",
    "What is Your Gender": "Male",
    "1. Have You Been Feeling Sad Most of The Time During The Day, Especially Over The Last Two Weeks ?": "Yes",
    "2. Do You Feel Unhappy Even When You Do Your Favorite Activities ?": "Yes",
    "3. Do You Feel Tired All The Time , Do You Find It Difficulty to Complete Day Today Usual Activity ?": "No",
    "4. Do you often catch yourself getting distracted while working on something important?": "Sometimes",
    "5. Do you often feel that you're not as capable as your friends, even when you try your best?": "Yes",
    "6. Do you often feel guilty about things that aren't your fault or feel worthless?": "No",
    "7. Do you feel as if things won’t get better no matter what you do?": "No",
    "8. Lately, have you felt so overwhelmed or hopeless that you thought about giving up on everything and even Life?": "No",
    "9. Have you been having trouble falling asleep, waking early, or sleeping too much?": "Sometimes",
    "10. Have you lost your appetite or found that you’re eating much less than usual?": "No",
    "11. Do you find it hard to stay motivated with your office work or daily goals?": "Sometimes",
    "12. Have you recently lost interest in socializing with coworkers?": "No",
    "13. Do you find your sleep pattern irregular due to work-related thoughts?": "Sometimes",
    "14. How often do you feel hopeless about your work or future?": "Rarely"


}
df_row = pd.DataFrame([row])
missing = [c for c in raw_input_columns if c not in df_row.columns]
if missing:
    raise ValueError(f"Missing columns in input dict: {missing}")
df_row = df_row[raw_input_columns].copy()
X = pre.transform(df_row)
if hasattr(X, "toarray"):
    X = X.toarray()
pred = model.predict(X)[0]
conf = None
if hasattr(model, "predict_proba"):
    conf = float(model.predict_proba(X).max())
print("Prediction:", pred, "| Confidence:", conf)

Prediction: No Depression | Confidence: 0.7687902494639035


In [25]:
# Batch prediction from CSV template
csv_path = DATA_RAW / "sample_input_template.csv"
print("Using CSV:", csv_path)
df = pd.read_csv(csv_path)
missing = [c for c in raw_input_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in input CSV: {missing}")
df = df[raw_input_columns].copy()
X = pre.transform(df)
if hasattr(X, "toarray"):
    X = X.toarray()
preds = model.predict(X)
probs = model.predict_proba(X) if hasattr(model, "predict_proba") else None
out = df.copy()
out["prediction"] = preds
if probs is not None:
    if probs.shape[1] <= 5:
        for j in range(probs.shape[1]):
            out[f"prob_class_{j}"] = probs[:, j]
    out["confidence"] = probs.max(axis=1)
out_path = REPORTS / "predictions.csv"
out.to_csv(out_path, index=False)
print("Saved predictions to:", out_path)
out.head()

Using CSV: d:\Git\DepressionLevel\DepressionModelProject\data\raw\sample_input_template.csv


ValueError: Found array with 0 sample(s) (shape=(0, 13)) while a minimum of 1 is required by SimpleImputer.

In [24]:
from pathlib import Path
import pandas as pd
import joblib

PROJ = Path.cwd().parents[0]
DATA_RAW = PROJ / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Load preprocessor to recover original input schema
pre = joblib.load(PROJ / "models" / "preprocessor.joblib")
num_cols = pre.transformers_[0][2]
cat_cols = pre.transformers_[1][2]
raw_input_columns = list(num_cols) + list(cat_cols)

template_path = DATA_RAW / "sample_input_template.csv"
pd.DataFrame(columns=raw_input_columns).to_csv(template_path, index=False, encoding="utf-8-sig")
print("Created template at:", template_path)


Created template at: d:\Git\DepressionLevel\DepressionModelProject\data\raw\sample_input_template.csv
